In [3]:
# US presidential election prediction 

In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import statsmodels

In [63]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn import linear_model
from lightgbm import LGBMRegressor


In [48]:
demo = pd.read_csv('/Users/mwilson89/Documents/GitHub/surveys/data/temp/demo_data_orig.csv')# Smith College machine
demo = demo.fillna(0)

In [49]:
# summarize demographics in a simple yet meaningful way

#demo['edu'] = (3*demo['deg']+2*demo['ass']+demo['elem'])/(3*demo['elem']+3*demo['ass']+3*demo['deg'])
#demo['race'] = (demo['white']-demo['black'])/(demo['white']+demo['black']+demo['nat']+demo['aapi']+demo['mixed'])
demo['malefr'] = demo['male']/(demo['male']+demo['female'])
demo['femalefr'] = demo['female']/(demo['male']+demo['female'])
demo['whitefr'] = demo['white']/(demo['white']+demo['black']+demo['nat']+demo['aapi']+demo['mixed'])
demo['blackfr'] = demo['black']/(demo['white']+demo['black']+demo['nat']+demo['aapi']+demo['mixed'])
demo['natfr'] = demo['nat']/(demo['white']+demo['black']+demo['nat']+demo['aapi']+demo['mixed'])
demo['aapifr'] = demo['aapi']/(demo['white']+demo['black']+demo['nat']+demo['aapi']+demo['mixed'])
demo['mixedfr'] = demo['mixed']/(demo['white']+demo['black']+demo['nat']+demo['aapi']+demo['mixed'])
demo['elemfr'] = demo['elem']/(demo['elem']+demo['ass']+demo['deg'])
demo['assfr'] = demo['ass']/(demo['elem']+demo['ass']+demo['deg'])
demo['degfr'] = demo['deg']/(demo['elem']+demo['ass']+demo['deg'])

In [50]:
# historical voting data

vote = pd.read_csv('/Users/mwilson89/Documents/GitHub/election_forecast/data/temp/vote_data_clean_algara.csv') 
vote = vote.fillna(0)

In [51]:
demo.shape, vote.shape

((57730, 26), (59099, 10))

In [52]:
# fix the year format so we can work out which ones to average over

demo['years'] = demo['year'].str.split(pat='-')
def getyear1(row):
    return row['years'][0]
def getyear2(row):
    if(len(row['years'])>1):
        return row['years'][1]
    else:
        return row['years'][0]
demo['year1'] = demo.apply(getyear1, axis=1).astype(int)
demo['year2'] = demo.apply(getyear2, axis=1).astype(int)
demo['pres_year'] = 4*demo['year2'].floordiv(4)
cols_to_drop = ['year','years','year1','year2']
demo = demo.drop(cols_to_drop, axis=1)

# now drop a lot more, maybe too many

#cols_to_drop = ['white','black','nat','aapi','mixed','elem','ass','deg','male','female']
cols_to_drop = ['male','female','white','black','nat','aapi','mixed','elem','ass','deg']
demo = demo.drop(cols_to_drop, axis=1)

# remove Puerto Rico etc

demo = demo[demo['state']<57]

In [53]:
demo

,state,county,inc,mage,fage,malefr,femalefr,whitefr,blackfr,natfr,aapifr,mixedfr,elemfr,assfr,degfr,pres_year
0,1,1,5774.0,26.3,29.4,0.491274,0.508726,0.770371,0.223986,0.003008,0.002635,0.000000,0.233888,0.644725,0.121388,1980
1,1,3,5960.0,29.4,31.8,0.488301,0.511699,0.840274,0.153404,0.005506,0.000816,0.000000,0.215371,0.664072,0.120557,1980
2,1,5,4544.0,27.5,32.4,0.471320,0.528680,0.551960,0.444566,0.000848,0.002626,0.000000,0.371133,0.536617,0.092251,1980
3,1,7,4859.0,27.7,30.8,0.485149,0.514851,0.765757,0.233734,0.000127,0.000382,0.000000,0.349332,0.601653,0.049015,1980
4,1,9,5213.0,30.1,32.5,0.494885,0.505115,0.978546,0.018134,0.002689,0.000631,0.000000,0.321950,0.625185,0.052865,1980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57647,56,37,40988.0,37.6,37.2,0.512038,0.487962,0.824056,0.011918,0.007658,0.063777,0.092591,0.029051,0.767191,0.203758,2020
57648,56,39,75541.0,39.0,40.5,0.526501,0.473499,0.804564,0.005865,0.001798,0.123041,0.064732,0.023670,0.364133,0.612197,2020
57649,56,41,34568.0,36.6,37.0,0.504683,0.495317,0.909634,0.002572,0.000971,0.024994,0.061830,0.022186,0.764622,0.213192,2020
57650,56,43,36030.0,44.0,44.4,0.502206,0.497794,0.825636,0.001816,0.005189,0.028931,0.138428,0.031449,0.748228,0.220324,2020


In [54]:
# average the data covering a given presidential year - can probably do better
# no data for 2004 so remove everything

#demo = demo[demo['pres_year']!=2004]
#vote = vote[vote['year']!=2004]
#pres_years = [2000,2004,2008,2012,2016,2020]

demo = demo.groupby(['pres_year','state','county']).mean().reset_index()

# get ready for merging with vote

demo['year'] = demo['pres_year']
demo['fips'] = 1000*demo['state']+demo['county']
demo = demo.drop(['pres_year','state','county'],axis=1)
print(demo.shape)

vote['fips'] = vote['fips'].astype(int)
vote['demfr'] = vote['DEMOCRAT']/vote['totalvotes']
vote['repfr'] = vote['REPUBLICAN']/vote['totalvotes']
vote['othfr'] = vote['OTHER']/vote['totalvotes']
vote['lagdemfr'] = vote['lagDEM']/(vote['lagDEM']+vote['lagREP']+vote['lagOTH'])
vote['lagrepfr'] = vote['lagREP']/(vote['lagDEM']+vote['lagREP']+vote['lagOTH'])
vote['lagothfr'] = vote['lagOTH']/(vote['lagDEM']+vote['lagREP']+vote['lagOTH'])
vote = vote.drop(['DEMOCRAT','REPUBLICAN','OTHER','totalvotes','lagDEM','lagREP','lagOTH'],axis=1)
vote = vote.fillna(0)
print(vote.shape)

(22001, 15)
(59099, 9)


In [55]:
def ml(mod):
    mod.fit(X_train, y_train)
    y_pred = mod.predict(X_test)
    mae = mean_absolute_error(y_true=y_test,y_pred=y_pred)
    mse = mean_squared_error(y_true=y_test,y_pred=y_pred) 
    r2=r2_score(y_true=y_test,y_pred=y_pred)
    print("Model : ",mod)
    print("R^2: ",r2)
    print("MAE: ",mae)
    print("MSE: ",mse)


In [66]:
# prediction using vote history only
#vote_train = vote[(vote['year']==2016)]
vote_train = vote[(vote['year']<2020)  & (vote['year']>1996)]
vote_test = vote[vote['year']==2020]
cols_to_drop1 = ['year','state','fips','demfr','repfr','othfr','lagothfr']
X_train = vote_train.drop(cols_to_drop1,axis=1)
X_test = vote_test.drop(cols_to_drop1,axis=1)
y_train = vote_train['demfr']
y_test = vote_test['demfr']

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
#from sklearn.model_selection import train_test_split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25)

#X_train.shape, X_test.shape, y_train.shape, y_test.shape

mods = [LinearRegression(),Lasso(),RandomForestRegressor(),HistGradientBoostingRegressor(), LGBMRegressor()]
for mod in mods:
    ml(mod)


Model :  LinearRegression()
R^2:  0.8838643248983326
MAE:  0.04631498843342438
MSE:  0.0029787886322482416
Model :  Lasso()
R^2:  -0.08737975289148991
MAE:  0.14056452459176647
MSE:  0.027890434562975783
Model :  RandomForestRegressor()
R^2:  0.8909819745914352
MAE:  0.04161107598614888
MSE:  0.002796226521375945
Model :  HistGradientBoostingRegressor()
R^2:  0.8965626411159761
MAE:  0.041595455627953665
MSE:  0.0026530868187038953
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000281 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 15569, number of used features: 2
[LightGBM] [Info] Start training from score 0.380274
Model :  LGBMRegressor()
R^2:  0.8978427265555626
MAE:  0.04114833807080898
MSE:  0.0026202536349951952


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [57]:
# now add polling data
### add new predictors e.g. swing - not used here

def sw(v0,vs0,d,str): 
    if(str=="ups"):
        di <- d    
    elif(str=="prop"):
        di <- d*v0/vs0
    elif(str=="pw"):
        if d>0:
            di = d*(1-v0)/(1-vs0)
        else:
            di <- d*v0/vs0
    return (di) #  Estimate of district level swing

poll = pd.DataFrame({
    'year': [1980,1984,1988,1992,1996,2000,2004,2008,2012,2016,2020],
    'pollD':[0.410,0.406,0.457,0.430,0.492,0.48,0.469, 0.521 ,0.488,0.468, 0.512], 
    'pollR':[0.508,0.588,0.534,0.375,0.407,0.46, 0.489, 0.445,0.481,0.436, 0.440]
#    'pollDfracdiff': [0.062,-0.012,-0.011,0.052, -0.033, -0.020, 0.044],
#   'pollRfracdiff': [0.032,0.053,0.029,-0.044, 0.036, -0.045, 0.004]
})
poll['pollDfracdiff'] = poll['pollD'] - poll['pollD'].shift(1)
poll['pollRfracdiff'] = poll['pollR'] - poll['pollR'].shift(1)
# seems not to work well, so will ignore for now or at least until we can get more data

vp = pd.merge(vote,poll[poll['year']>1996],on='year')
#vp = pd.merge(vote,poll,on='year')
vp = vp.fillna(0)

In [67]:
# prediction using vote history and polling data only
vp_train = vp[(vp['year']<2020) & (vp['year'] > 1996)]
vp_test = vp[vp['year']==2020]
cols_to_drop1 = ['state','fips','demfr','repfr','othfr','lagothfr']
#cols_to_drop2 = ['lagrepfr','pollR']
#cols_to_drop2 = ['pollDfracdiff','pollRfracdiff']
cols_to_drop2 = []
X_train = vp_train.drop(cols_to_drop1,axis=1)
X_train = X_train.drop(cols_to_drop2,axis=1)
X_test = vp_test.drop(cols_to_drop1,axis=1)
X_test = X_test.drop(cols_to_drop2,axis=1)
y_train = vp_train['demfr']
y_test = vp_test['demfr']
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
#X_train.shape, X_test.shape, y_train.shape, y_test.shape

mods = [LinearRegression(),Lasso(), RandomForestRegressor(),HistGradientBoostingRegressor(), LGBMRegressor()]
for mod in mods:
    ml(mod)

Model :  LinearRegression()
R^2:  0.9722548762261042
MAE:  0.018791716298836667
MSE:  0.000711640580946815
Model :  Lasso()
R^2:  -0.08737975289148991
MAE:  0.14056452459176647
MSE:  0.027890434562975783
Model :  RandomForestRegressor()
R^2:  0.9466826973004959
MAE:  0.028238331273548644
MSE:  0.0013675468373037505
Model :  HistGradientBoostingRegressor()
R^2:  0.9523027026329821
MAE:  0.026286676431435934
MSE:  0.0012233981251795112
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000226 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 540
[LightGBM] [Info] Number of data points in the train set: 15569, number of used features: 7
[LightGBM] [Info] Start training from score 0.380274
Model :  LGBMRegressor()
R^2:  0.9562783052666051
MAE:  0.02527784805667547
MSE:  0.0011214270476358952


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [59]:
poly = PolynomialFeatures(degree=2)
X_poly_train = poly.fit_transform(X_train)
X_poly_test = poly.fit_transform(X_test)
mod = LinearRegression()
mod.fit(X_poly_train,y_train)
y_pred = mod.predict(X_poly_test)
mae = mean_absolute_error(y_true=y_test,y_pred=y_pred)
mse = mean_squared_error(y_true=y_test,y_pred=y_pred) 
r2=r2_score(y_true=y_test,y_pred=y_pred)
print("Model : ",mod)
print("R^2: ",r2)
print("MAE: ",mae)
print("MSE: ",mse)

NameError: name 'PolynomialFeatures' is not defined

In [ ]:
#mod.score(X_train,y_train), mod.coef_, mod.intercept_
import statsmodels.api as sm
import statsmodels.formula.api as smf
#results = smf.ols('demfr ~  lagdemfr + lagrepfr + pollD + pollR + pollDfracdiff + pollRfracdiff', data=vp).fit()
#results = smf.ols('demfr ~  lagdemfr + lagrepfr + pollD + pollR + pollDfracdiff + pollRfracdiff', data=vp).fit_regularized(method='elastic_net', alpha=0.1, L1_wt=1.0)
results = smf.ols('demfr ~ lagdemfr + pollD + lagrepfr + femalefr*fage +whitefr+blackfr+aapifr', data=df).fit()
print(results.summary())
sns.scatterplot(x=results.fittedvalues, y=results.resid)

#sm.graphics.plot_leverage_resid2(results)

In [ ]:
# messing around

from sklearn.preprocessing import PolynomialFeatures
vp_train = vp[(vp['year']<2020)]
vp_test = vp[vp['year']==2020]

X_train = vp_train[['lagdemfr','pollD','lagrepfr','pollR']]
X_test = vp_test[['lagdemfr', 'pollD','lagrepfr','pollR']]
y_train = vp_train['demfr']
y_test = vp_test['demfr']

poly = PolynomialFeatures(degree=1)
X_poly_train = poly.fit_transform(X_train)
X_poly_test = poly.fit_transform(X_test)
mod = LinearRegression()
mod.fit(X_poly_train,y_train)
y_pred = mod.predict(X_poly_test)
mae = mean_absolute_error(y_true=y_test,y_pred=y_pred)
mse = mean_squared_error(y_true=y_test,y_pred=y_pred) 
r2=r2_score(y_true=y_test,y_pred=y_pred)
print("Model : ",mod)
print("R^2: ",r2)
print("MAE: ",mae)
print("MSE: ",mse)


In [ ]:
df = pd.merge(vp,demo, on=['year','fips'])
df = df.fillna(0)


#df['pollupspredD'] = df['lagdemfr']+df['pollDfracdiff'] # temp fix to avoid computing overall swing
#df['pollupspredR'] = df['lagrepfr']+df['pollRfracdiff'] # temp fix to avoid computing overall swing

In [60]:
vp_train

,year,state,fips,demfr,repfr,othfr,lagdemfr,lagrepfr,lagothfr,pollD,pollR,pollDfracdiff,pollRfracdiff
0,2000,AL,1001,0.287192,0.696943,0.015865,0.325185,0.616587,0.058229,0.480,0.460,-0.012,0.053
1,2004,AL,1001,0.236940,0.756735,0.006324,0.287192,0.696943,0.015865,0.469,0.489,-0.011,0.029
2,2008,AL,1001,0.257730,0.736136,0.006133,0.236940,0.756735,0.006324,0.521,0.445,0.052,-0.044
3,2012,AL,1001,0.265424,0.724941,0.009636,0.257730,0.736136,0.006133,0.488,0.481,-0.033,0.036
4,2016,AL,1001,0.237697,0.727666,0.034637,0.265424,0.724941,0.009636,0.468,0.436,-0.020,-0.045
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18676,2000,WY,56045,0.146732,0.823856,0.029412,0.274417,0.555451,0.170132,0.480,0.460,-0.012,0.053
18677,2004,WY,56045,0.170991,0.807488,0.021521,0.146732,0.823856,0.029412,0.469,0.489,-0.011,0.029
18678,2008,WY,56045,0.193929,0.771589,0.034483,0.170991,0.807488,0.021521,0.521,0.445,0.052,-0.044
18679,2012,WY,56045,0.125633,0.839833,0.034534,0.193929,0.771589,0.034483,0.488,0.481,-0.033,0.036


In [61]:
# add interaction term

df = df.dropna()
#df['male'] = df['malefr']*df['mage']
#df['female'] = df['femalefr']*df['fage']

NameError: name 'df' is not defined

In [932]:
# prediction using vote history, polling data and demographics
df_train = df[(df['year']<2020) &(df['year']>1996)]
df_test = df[df['year']==2020]
cols_to_drop1 = ['state','fips','demfr','repfr','othfr','lagothfr','femalefr','mixedfr']
#cols_to_drop2 = ['mage','fage','malefr','femalefr']
cols_to_drop2 = []
X_train = df_train.drop(cols_to_drop1,axis=1)
X_train = X_train.drop(cols_to_drop2,axis=1)
X_test = df_test.drop(cols_to_drop1,axis=1)
X_test = X_test.drop(cols_to_drop2,axis=1)
y_train = df_train['demfr']
y_test = df_test['demfr']
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
#X_train.shape, X_test.shape, y_train.shape, y_test.shape

mods = [LinearRegression(),RandomForestRegressor(),XGBRegressor()]
for mod in mods:
    ml(mod)

Model :  LinearRegression()
R^2:  0.5046107212801512
MAE:  0.10536954176040597
MSE:  0.012706344977084083
Model :  RandomForestRegressor()
R^2:  0.8017006814324285
MAE:  0.06260451616769584
MSE:  0.005086221399363732
Model :  XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)
R^2:  0.7844824767703006

In [928]:
# prediction using vote history and demographics only

df = pd.merge(vote,demo, on=['year','fips'])
df = df.fillna(0)
df['male'] = df['malefr']*df['mage']
df['female'] = df['femalefr']*df['fage']


df_train = df[(df['year']<2020)]
df_test = df[df['year']==2020]
cols_to_drop1 = ['state','fips','demfr','repfr','othfr','lagothfr']
cols_to_drop2 = ['mage','fage','malefr','femalefr']
X_train = df_train.drop(cols_to_drop1,axis=1)
X_train = X_train.drop(cols_to_drop2,axis=1)
X_test = df_test.drop(cols_to_drop1,axis=1)
X_test = X_test.drop(cols_to_drop2,axis=1)
y_train = df_train['demfr']
y_test = df_test['demfr']
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
#X_train.shape, X_test.shape, y_train.shape, y_test.shape

mods = [LinearRegression(),RandomForestRegressor(),XGBRegressor()]
for mod in mods:
    ml(mod)

Model :  LinearRegression()
R^2:  0.6919436826503019
MAE:  0.07544461236281215
MSE:  0.007901402005974667
Model :  RandomForestRegressor()
R^2:  0.8047928029914401
MAE:  0.06250697582668038
MSE:  0.005006910915815501
Model :  XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)
R^2:  0.7800149821802763